# Study 3 Chicago Face Database + CLIP Pilot Study

- The pilot used `CFD Version 3.0` neutral-expression images as the retrieval gallery.
- Images were indexed using the sentence-transformers CLIP model `clip-ViT-B-32`.
- Both image and text embeddings were `L2-normalized`, and retrieval used `cosine similarity` via inner product.
- Proxy descriptions were generated from CFD age, race, and gender metadata using a fixed natural-language template.

The pilot study is designed to answer two feasibility questions:

1. Can CLIP retrieve the corresponding CFD target from a text description?
2. Can CLIP generate stable top-5 candidate fillers for each target/description?

Environment can be created with the code here in PowerShell:

```powershell
python -m pip install -e ".[cfd-pilot]"
python -c "import sentence_transformers, torch, openpyxl; print('CLIP pilot deps OK')"
```

## 0. Paths And Dependency Check

Prepares the output directory, and checks the dependencies required for the study.

In [ ]:
from pathlib import Path
import importlib.util
import json
import sys

import pandas as pd

# The current working directory may be either the repository root or the studies folder.
cwd = Path.cwd().resolve()
ROOT = cwd if (cwd / "src" / "pyWitnessAI").exists() else cwd.parent
SRC = ROOT / "src"
OUT = ROOT / "build" / "study3_cfd_clip_pilot_outputs"

if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

OUT.mkdir(parents=True, exist_ok=True)

print("ROOT:", ROOT)
print("OUT:", OUT)

for package in ["sentence_transformers", "torch", "openpyxl", "PIL", "pandas"]:
    print(f"{package:22s}", "OK" if importlib.util.find_spec(package) else "MISSING")

In [ ]:
from pyWitnessAI.cfd_clip_pilot import (
    ClipIndex,
    build_cfd_manifest,
    build_filler_sets,
    build_proxy_descriptions,
    evaluate_retrieval,
)
from pyWitnessAI.cfd_clip_pilot.clip_backend import SentenceTransformerClipEncoder

CFD_IMAGE_DIR = ROOT / "data" / "CFD Version 3.0" / "Images"
CFD_METADATA_PATH = ROOT / "data" / "CFD Version 3.0" / "CFD 3.0 Norming Data and Codebook.xlsx"

MODEL_NAME = "clip-ViT-B-32"
DEVICE = "cpu"  # Change to "cuda" if a CUDA-enabled PyTorch installation is available.
SEED = 2026

print("CFD_IMAGE_DIR exists:", CFD_IMAGE_DIR.exists())
print("CFD_METADATA_PATH exists:", CFD_METADATA_PATH.exists())

## 1. Build The Manifest

`manifest.csv` is the master index for the CFD gallery. Each row corresponds to one neutral face image and is merged with metadata from the official CFD norming workbook.

Core columns include:

- `image_id`: full image identifier, for example `CFD-AF-200-228-N`
- `target_id`: model identifier aligned with the CFD workbook, for example `AF-200`
- `expression`: expression code; this pilot keeps neutral images (`N`) by default
- `image_path`: image file path
- `gender`, `race`, `age`: standardized helper fields derived from the CFD metadata

The manifest also preserves the original CFD ratings and facial measurements for later diagnostic or fairness-oriented analyses.

In [3]:
manifest = build_cfd_manifest(
    image_dir=CFD_IMAGE_DIR,
    metadata_path=CFD_METADATA_PATH,
    neutral_only=True,
)

manifest_path = OUT / "manifest.csv"
manifest.to_csv(manifest_path, index=False)

print("Saved:", manifest_path)
print("shape:", manifest.shape)
manifest[["image_id", "target_id", "expression", "gender", "race", "age", "image_path"]].head()

Saved: D:\PhD\Software\pyWitnessAI\build\study3_cfd_clip_pilot_outputs\manifest.csv
shape: (831, 161)


,image_id,target_id,expression,gender,race,age,image_path
0,CFD-AF-200-228-N,AF-200,N,female,Asian,32.571429,D:\PhD\Software\pyWitnessAI\data\CFD Version 3...
1,CFD-AF-201-060-N,AF-201,N,female,Asian,23.666667,D:\PhD\Software\pyWitnessAI\data\CFD Version 3...
2,CFD-AF-202-122-N,AF-202,N,female,Asian,24.448276,D:\PhD\Software\pyWitnessAI\data\CFD Version 3...
3,CFD-AF-203-077-N,AF-203,N,female,Asian,22.758621,D:\PhD\Software\pyWitnessAI\data\CFD Version 3...
4,CFD-AF-204-067-N,AF-204,N,female,Asian,30.137931,D:\PhD\Software\pyWitnessAI\data\CFD Version 3...


In [4]:
summary_manifest = {
    "rows": len(manifest),
    "unique_targets": int(manifest["target_id"].nunique()),
    "missing_gender": int(manifest["gender"].isna().sum()),
    "missing_race": int(manifest["race"].isna().sum()),
    "missing_age": int(manifest["age"].isna().sum()),
}
summary_manifest

{'rows': 831,
 'unique_targets': 827,
 'missing_gender': 1,
 'missing_race': 1,
 'missing_age': 1}

In [5]:
manifest["race"].fillna("<missing>").value_counts().rename("n")

race
Black          197
White          183
Indian         145
Asian          109
Latino         108
Multiracial     88
<missing>        1
Name: n, dtype: int64

## 2. Generate Proxy Descriptions

`queries_complete.csv` is not a set of real eyewitness descriptions. It is a proxy query table generated from CFD metadata.

For example, the row for `AF-200` uses:

- `age = 32.57...`
- `race = Asian`
- `gender = female`

and renders the following CLIP-friendly sentence:

`A neutral frontal face photograph of around 32.6 years old asian female person.`

`required_fields=["age", "gender", "race"]` excludes targets with incomplete metadata.

In [7]:
queries = build_proxy_descriptions(
    manifest,
    required_fields=["age", "gender", "race"],
)

queries_path = OUT / "queries_complete.csv"
queries.to_csv(queries_path, index=False)

print("Saved:", queries_path)
print("shape:", queries.shape)
queries[["query_id", "target_id", "image_id", "description"]].head(10)

Saved: D:\PhD\Software\pyWitnessAI\build\study3_cfd_clip_pilot_outputs\queries_complete.csv
shape: (826, 164)


,query_id,target_id,image_id,description
0,AF-200,AF-200,CFD-AF-200-228-N,A neutral frontal face photograph of around 32...
1,AF-201,AF-201,CFD-AF-201-060-N,A neutral frontal face photograph of around 23...
2,AF-202,AF-202,CFD-AF-202-122-N,A neutral frontal face photograph of around 24...
3,AF-203,AF-203,CFD-AF-203-077-N,A neutral frontal face photograph of around 22...
4,AF-204,AF-204,CFD-AF-204-067-N,A neutral frontal face photograph of around 30...
5,AF-205,AF-205,CFD-AF-205-155-N,A neutral frontal face photograph of around 26...
6,AF-206,AF-206,CFD-AF-206-079-N,A neutral frontal face photograph of around 26...
7,AF-207,AF-207,CFD-AF-207-023-N,A neutral frontal face photograph of around 28...
8,AF-208,AF-208,CFD-AF-208-003-N,A neutral frontal face photograph of around 28...
9,AF-209,AF-209,CFD-AF-209-006-N,A neutral frontal face photograph of around 22...


## 3. Smoke Test: Random Subsample

The PowerShell workflow used `--max-images 50`, which takes the first 50 rows after path sorting. In CFD 3.0, that can over-sample `CFD-INDIA` because of path ordering.

Here, the smoke test draws a random set of 50 metadata-complete targets from the full manifest. This is a better workflow check, but it should still not be interpreted as the main performance estimate.

In [8]:
SMOKE_N = 50

eligible = manifest.dropna(subset=["age", "gender", "race"]).copy()
manifest_smoke = (
    eligible.sample(n=min(SMOKE_N, len(eligible)), random_state=SEED)
    .sort_values(["race", "gender", "target_id"])
    .reset_index(drop=True)
)

manifest_smoke_path = OUT / "manifest_smoke_random.csv"
manifest_smoke.to_csv(manifest_smoke_path, index=False)

queries_smoke = build_proxy_descriptions(
    manifest_smoke,
    required_fields=["age", "gender", "race"],
)
queries_smoke_path = OUT / "queries_smoke_random.csv"
queries_smoke.to_csv(queries_smoke_path, index=False)

print("smoke manifest:", manifest_smoke.shape, manifest_smoke_path)
print("smoke queries:", queries_smoke.shape, queries_smoke_path)
manifest_smoke.groupby(["race", "gender"]).size().rename("n").reset_index()

smoke manifest: (50, 161) D:\PhD\Software\pyWitnessAI\build\study3_cfd_clip_pilot_outputs\manifest_smoke_random.csv
smoke queries: (50, 164) D:\PhD\Software\pyWitnessAI\build\study3_cfd_clip_pilot_outputs\queries_smoke_random.csv


,race,gender,n
0,Asian,female,2
1,Asian,male,4
2,Black,female,9
3,Black,male,8
4,Indian,female,1
5,Indian,male,6
6,Latino,female,2
7,Latino,male,1
8,Multiracial,female,4
9,Multiracial,male,2


## 4. Build And Evaluate The Smoke CLIP Index

This section loads CLIP and encodes the smoke-test images. The first run may download model weights.

In [9]:
encoder = SentenceTransformerClipEncoder(model_name=MODEL_NAME, device=DEVICE)

clip_index_smoke = ClipIndex.build(
    manifest=manifest_smoke,
    encoder=encoder,
    batch_size=16,
    show_progress=True,
)

smoke_index_dir = OUT / "clip_index_smoke_random"
clip_index_smoke.save(smoke_index_dir)
print("Saved smoke index:", smoke_index_dir)

Encoding images: 100%|██████████| 4/4 [00:04<00:00,  1.15s/it]

Saved smoke index: D:\PhD\Software\pyWitnessAI\build\study3_cfd_clip_pilot_outputs\clip_index_smoke_random


In [10]:
retrieval_smoke, per_query_smoke, summary_smoke = evaluate_retrieval(
    index=clip_index_smoke,
    queries=queries_smoke,
    encoder=encoder,
    top_k=20,
)

eval_smoke_dir = OUT / "evaluation_smoke_random"
eval_smoke_dir.mkdir(parents=True, exist_ok=True)
retrieval_smoke.to_csv(eval_smoke_dir / "retrieval_results.csv", index=False)
per_query_smoke.to_csv(eval_smoke_dir / "per_query_metrics.csv", index=False)
(eval_smoke_dir / "summary.json").write_text(json.dumps(summary_smoke, indent=2), encoding="utf-8")

summary_smoke

{'n_queries': 50,
 'top_k': 20,
 'found_in_top_k': 49,
 'not_found_in_top_k': 1,
 'mrr_with_misses_as_zero': 0.404053835053835,
 'hit_rate_top_1': 0.18,
 'hit_rate_top_5': 0.74,
 'hit_rate_top_10': 0.88,
 'hit_rate_top_20': 0.98,
 'mean_rank_found_only': 4.346938775510204,
 'median_rank_found_only': 3.0}

## 5. Build Or Load The Full CLIP Index

If the full index already exists from a PowerShell run, this cell loads it. To force a rebuild, set `REBUILD_FULL_INDEX = True`.

In [11]:
REBUILD_FULL_INDEX = False
full_index_dir = OUT / "clip_index"

if (full_index_dir / "image_embeddings.npy").exists() and not REBUILD_FULL_INDEX:
    clip_index = ClipIndex.load(full_index_dir)
    print("Loaded existing full index:", full_index_dir)
else:
    clip_index = ClipIndex.build(
        manifest=manifest,
        encoder=encoder,
        batch_size=32,
        show_progress=True,
    )
    clip_index.save(full_index_dir)
    print("Saved full index:", full_index_dir)

clip_index.metadata

Loaded existing full index: D:\PhD\Software\pyWitnessAI\build\study3_cfd_clip_pilot_outputs\clip_index


{'embedding_dim': 512,
 'embedding_sha256': '942677295a08c51c19e47565d431e94c47e5be8d82f700142df853ae84796dea',
 'image_count': 831,
 'model_name': 'clip-ViT-B-32'}

## 6. Full Retrieval Evaluation

This step retrieves the top-k CFD images for each proxy description and checks whether the true `target_id` appears among the returned results.

In [12]:
retrieval_results, per_query, summary = evaluate_retrieval(
    index=clip_index,
    queries=queries,
    encoder=encoder,
    top_k=50,
)

eval_dir = OUT / "evaluation"
eval_dir.mkdir(parents=True, exist_ok=True)
retrieval_results.to_csv(eval_dir / "retrieval_results.csv", index=False)
per_query.to_csv(eval_dir / "per_query_metrics.csv", index=False)
(eval_dir / "summary.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")

summary

{'n_queries': 826,
 'top_k': 50,
 'found_in_top_k': 466,
 'not_found_in_top_k': 360,
 'mrr_with_misses_as_zero': 0.0739268391506932,
 'hit_rate_top_1': 0.020581113801452784,
 'hit_rate_top_5': 0.1016949152542373,
 'hit_rate_top_10': 0.17554479418886199,
 'hit_rate_top_50': 0.5641646489104116,
 'mean_rank_found_only': 20.326180257510728,
 'median_rank_found_only': 19.0}

In [13]:
pd.DataFrame(
    {
        "metric": ["top_1", "top_5", "top_10", "top_50"],
        "hit_rate": [
            per_query["hit_top_1"].mean(),
            per_query["hit_top_5"].mean(),
            per_query["hit_top_10"].mean(),
            per_query["hit_top_50"].mean(),
        ],
        "n_hit": [
            int(per_query["hit_top_1"].sum()),
            int(per_query["hit_top_5"].sum()),
            int(per_query["hit_top_10"].sum()),
            int(per_query["hit_top_50"].sum()),
        ],
    }
)

,metric,hit_rate,n_hit
0,top_1,0.020581,17
1,top_5,0.101695,84
2,top_10,0.175545,145
3,top_50,0.564165,466


In [14]:
per_query[["query_id", "target_id", "description", "target_rank", "hit_top_50"]].head(10)

,query_id,target_id,description,target_rank,hit_top_50
0,AF-200,AF-200,A neutral frontal face photograph of around 32...,31.0,True
1,AF-201,AF-201,A neutral frontal face photograph of around 23...,15.0,True
2,AF-202,AF-202,A neutral frontal face photograph of around 24...,NaN,False
3,AF-203,AF-203,A neutral frontal face photograph of around 22...,25.0,True
4,AF-204,AF-204,A neutral frontal face photograph of around 30...,36.0,True
5,AF-205,AF-205,A neutral frontal face photograph of around 26...,9.0,True
6,AF-206,AF-206,A neutral frontal face photograph of around 26...,7.0,True
7,AF-207,AF-207,A neutral frontal face photograph of around 28...,NaN,False
8,AF-208,AF-208,A neutral frontal face photograph of around 28...,NaN,False
9,AF-209,AF-209,A neutral frontal face photograph of around 22...,NaN,False


## 7. Metadata-Stratified Diagnostics

These summaries are pilot diagnostics, not formal fairness conclusions. The descriptions are metadata-generated proxy text rather than real eyewitness descriptions.

In [15]:
per_query.groupby("gender").agg(
    n=("query_id", "count"),
    hit_top_5=("hit_top_5", "mean"),
    hit_top_10=("hit_top_10", "mean"),
    hit_top_50=("hit_top_50", "mean"),
    median_rank=("target_rank", "median"),
).reset_index()

,gender,n,hit_top_5,hit_top_10,hit_top_50,median_rank
0,female,421,0.087886,0.163895,0.529691,19.0
1,male,405,0.116049,0.187654,0.600000,18.0


In [16]:
per_query.groupby("race").agg(
    n=("query_id", "count"),
    hit_top_5=("hit_top_5", "mean"),
    hit_top_10=("hit_top_10", "mean"),
    hit_top_50=("hit_top_50", "mean"),
    median_rank=("target_rank", "median"),
).sort_values("hit_top_50", ascending=False).reset_index()

,race,n,hit_top_5,hit_top_10,hit_top_50,median_rank
0,Asian,109,0.137615,0.247706,0.752294,18.0
1,Latino,108,0.166667,0.212963,0.675926,20.0
2,Black,197,0.091371,0.172589,0.598985,20.0
3,Indian,141,0.134752,0.205674,0.581560,15.5
4,White,183,0.076503,0.153005,0.508197,15.0
5,Multiracial,88,0.000000,0.045455,0.204545,19.0


## 8. Export Top-5 Fillers

Excludes the same `target_id` by default so that the target is not selected as a filler.

In [23]:
fillers = build_filler_sets(
    index=clip_index,
    queries=queries,
    encoder=encoder,
    top_k=50,
    filler_count=5,
)

fillers_path = OUT / "filler_sets.csv"
fillers.to_csv(fillers_path, index=False)

print("Saved:", fillers_path)
print("shape:", fillers.shape)
fillers.head(10)

Saved: D:\PhD\Software\pyWitnessAI\build\study3_cfd_clip_pilot_outputs\filler_sets.csv
shape: (4130, 10)


,query_index,query_id,query_target_id,filler_position,candidate_rank,clip_score,filler_image_id,filler_target_id,filler_image_path,description
0,0,AF-200,AF-200,1,1,0.386120,CFD-AF-216-106-N,AF-216,data\CFD Version 3.0\Images\CFD\AF-216\CFD-AF-...,A neutral frontal face photograph of around 32...
1,0,AF-200,AF-200,2,2,0.381737,CFD-AF-256-160-N,AF-256,data\CFD Version 3.0\Images\CFD\AF-256\CFD-AF-...,A neutral frontal face photograph of around 32...
2,0,AF-200,AF-200,3,3,0.379651,CFD-AF-211-066-N,AF-211,data\CFD Version 3.0\Images\CFD\AF-211\CFD-AF-...,A neutral frontal face photograph of around 32...
3,0,AF-200,AF-200,4,4,0.379582,CFD-AF-229-160-N,AF-229,data\CFD Version 3.0\Images\CFD\AF-229\CFD-AF-...,A neutral frontal face photograph of around 32...
4,0,AF-200,AF-200,5,5,0.376854,CFD-AF-205-155-N,AF-205,data\CFD Version 3.0\Images\CFD\AF-205\CFD-AF-...,A neutral frontal face photograph of around 32...
5,1,AF-201,AF-201,1,1,0.373967,CFD-AF-219-106-N,AF-219,data\CFD Version 3.0\Images\CFD\AF-219\CFD-AF-...,A neutral frontal face photograph of around 23...
6,1,AF-201,AF-201,2,2,0.370284,CFD-AF-211-066-N,AF-211,data\CFD Version 3.0\Images\CFD\AF-211\CFD-AF-...,A neutral frontal face photograph of around 23...
7,1,AF-201,AF-201,3,3,0.366538,CFD-AF-222-134-N,AF-222,data\CFD Version 3.0\Images\CFD\AF-222\CFD-AF-...,A neutral frontal face photograph of around 23...
8,1,AF-201,AF-201,4,4,0.366428,CFD-AF-216-106-N,AF-216,data\CFD Version 3.0\Images\CFD\AF-216\CFD-AF-...,A neutral frontal face photograph of around 23...
9,1,AF-201,AF-201,5,5,0.363173,CFD-AF-229-160-N,AF-229,data\CFD Version 3.0\Images\CFD\AF-229\CFD-AF-...,A neutral frontal face photograph of around 23...


In [24]:
# Check whether every query received exactly five fillers.
fillers.groupby("query_id").size().value_counts().rename("n_queries")

5    826
Name: n_queries, dtype: int64

## 9. Inspect One Target

Example: `AF-200`.

In [26]:
TARGET_ID = "AF-200"

print("Query row")
display(queries.loc[queries["query_id"] == TARGET_ID, ["query_id", "target_id", "image_id", "description"]])

print("Evaluation row")
display(per_query.loc[per_query["query_id"] == TARGET_ID, ["query_id", "target_id", "target_rank", "hit_top_50"]])

print("Top retrievals")
display(retrieval_results.loc[retrieval_results["query_query_id"] == TARGET_ID].head(10))

print("Selected fillers")
display(fillers.loc[fillers["query_id"] == TARGET_ID])

Query row


,query_id,target_id,image_id,description
0,AF-200,AF-200,CFD-AF-200-228-N,A neutral frontal face photograph of around 32...


Evaluation row


,query_id,target_id,target_rank,hit_top_50
0,AF-200,AF-200,31.0,True


Top retrievals


,query_index,rank,image_index,image_id,target_id,image_path,clip_score,query_query_id,query_target_id,query_image_id,query_description,is_target
0,0,1,16,CFD-AF-216-106-N,AF-216,data\CFD Version 3.0\Images\CFD\AF-216\CFD-AF-...,0.386120,AF-200,AF-200,CFD-AF-200-228-N,A neutral frontal face photograph of around 32...,False
1,0,2,56,CFD-AF-256-160-N,AF-256,data\CFD Version 3.0\Images\CFD\AF-256\CFD-AF-...,0.381737,AF-200,AF-200,CFD-AF-200-228-N,A neutral frontal face photograph of around 32...,False
2,0,3,11,CFD-AF-211-066-N,AF-211,data\CFD Version 3.0\Images\CFD\AF-211\CFD-AF-...,0.379651,AF-200,AF-200,CFD-AF-200-228-N,A neutral frontal face photograph of around 32...,False
3,0,4,29,CFD-AF-229-160-N,AF-229,data\CFD Version 3.0\Images\CFD\AF-229\CFD-AF-...,0.379582,AF-200,AF-200,CFD-AF-200-228-N,A neutral frontal face photograph of around 32...,False
4,0,5,5,CFD-AF-205-155-N,AF-205,data\CFD Version 3.0\Images\CFD\AF-205\CFD-AF-...,0.376854,AF-200,AF-200,CFD-AF-200-228-N,A neutral frontal face photograph of around 32...,False
5,0,6,6,CFD-AF-206-079-N,AF-206,data\CFD Version 3.0\Images\CFD\AF-206\CFD-AF-...,0.376201,AF-200,AF-200,CFD-AF-200-228-N,A neutral frontal face photograph of around 32...,False
6,0,7,26,CFD-AF-226-251-N,AF-226,data\CFD Version 3.0\Images\CFD\AF-226\CFD-AF-...,0.375150,AF-200,AF-200,CFD-AF-200-228-N,A neutral frontal face photograph of around 32...,False
7,0,8,22,CFD-AF-222-134-N,AF-222,data\CFD Version 3.0\Images\CFD\AF-222\CFD-AF-...,0.375117,AF-200,AF-200,CFD-AF-200-228-N,A neutral frontal face photograph of around 32...,False
8,0,9,30,CFD-AF-230-193-N,AF-230,data\CFD Version 3.0\Images\CFD\AF-230\CFD-AF-...,0.374714,AF-200,AF-200,CFD-AF-200-228-N,A neutral frontal face photograph of around 32...,False
9,0,10,14,CFD-AF-214-139-N,AF-214,data\CFD Version 3.0\Images\CFD\AF-214\CFD-AF-...,0.373530,AF-200,AF-200,CFD-AF-200-228-N,A neutral frontal face photograph of around 32...,False


Selected fillers


,query_index,query_id,query_target_id,filler_position,candidate_rank,clip_score,filler_image_id,filler_target_id,filler_image_path,description
0,0,AF-200,AF-200,1,1,0.386120,CFD-AF-216-106-N,AF-216,data\CFD Version 3.0\Images\CFD\AF-216\CFD-AF-...,A neutral frontal face photograph of around 32...
1,0,AF-200,AF-200,2,2,0.381737,CFD-AF-256-160-N,AF-256,data\CFD Version 3.0\Images\CFD\AF-256\CFD-AF-...,A neutral frontal face photograph of around 32...
2,0,AF-200,AF-200,3,3,0.379651,CFD-AF-211-066-N,AF-211,data\CFD Version 3.0\Images\CFD\AF-211\CFD-AF-...,A neutral frontal face photograph of around 32...
3,0,AF-200,AF-200,4,4,0.379582,CFD-AF-229-160-N,AF-229,data\CFD Version 3.0\Images\CFD\AF-229\CFD-AF-...,A neutral frontal face photograph of around 32...
4,0,AF-200,AF-200,5,5,0.376854,CFD-AF-205-155-N,AF-205,data\CFD Version 3.0\Images\CFD\AF-205\CFD-AF-...,A neutral frontal face photograph of around 32...


## 10. Next step: Different Models, Real Human Descriptions, Add ArcFace

We can do comparisons:

- clip-ViT-B-32 vs ViT-L/14
- top_k = 20, 50, 100
- raw metadata proxy descriptions vs real human descriptions
- with vs without diversity filtering
- CLIP-only vs CLIP + ArcFace reranking

For a real-description pilot, prepare `queries_real.csv` with at least these columns:

- `query_id`: unique ID for each description
- `participant_id`: participant ID
- `target_id`: corresponding CFD target, for example `AF-200`
- `image_id`: corresponding CFD image ID; optional but recommended
- `description`: the participant's free-recall description

If the target is not in the CFD gallery, target-rank evaluation is not possible; in that case, export fillers and evaluate filler-description match with human ratings.

In [ ]:
real_template_path = OUT / "queries_real_template.csv"

template = pd.DataFrame(
    columns=["query_id", "participant_id", "target_id", "image_id", "description"]
)
template.to_csv(real_template_path, index=False)

print("Saved real-description template:", real_template_path)
template

Save real participant descriptions as:

`build/study3_cfd_clip_pilot_outputs/queries_real.csv`

Then uncomment and run the code below.

In [ ]:
# real_queries_path = OUT / "queries_real.csv"
# real_queries = pd.read_csv(real_queries_path)
#
# retrieval_real, per_query_real, summary_real = evaluate_retrieval(
#     index=clip_index,
#     queries=real_queries,
#     encoder=encoder,
#     top_k=50,
# )
#
# eval_real_dir = OUT / "evaluation_real"
# eval_real_dir.mkdir(parents=True, exist_ok=True)
# retrieval_real.to_csv(eval_real_dir / "retrieval_results.csv", index=False)
# per_query_real.to_csv(eval_real_dir / "per_query_metrics.csv", index=False)
# (eval_real_dir / "summary.json").write_text(json.dumps(summary_real, indent=2), encoding="utf-8")
#
# summary_real

In [ ]:
# fillers_real = build_filler_sets(
#     index=clip_index,
#     queries=real_queries,
#     encoder=encoder,
#     top_k=50,
#     filler_count=5,
# )
# fillers_real.to_csv(OUT / "filler_sets_real.csv", index=False)
# fillers_real.head(10)